# Experiment 01: Metric Sanity

Artificial subspace sanity checks for the new full/top alignment metrics. This notebook is not executed by default.

In [1]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "src" / "config.py").exists() and (candidate / "README.md").exists():
            return candidate
    raise RuntimeError("Could not locate repository root.")


REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repo root: {REPO_ROOT}")


Repo root: C:\Users\WWindows10\Documents\github_project\python-neural-network-research


In [2]:
import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable=None, *args, **kwargs):
        return iterable if iterable is not None else []

from src.metrics import (
    build_sampled_basis_matrix_from_indices,
    calculate_rank_metrics,
    calculate_subspace_alignment_from_matrices,
)


In [3]:
NUM_FREQS = 4
SEQ_LEN = 2048
TARGET_INDICES = np.arange(SEQ_LEN, dtype=np.int64)
THETAS = tuple(np.linspace(0.10 * np.pi, 0.70 * np.pi, NUM_FREQS))
ORTH_NOISE_SCALES = [0.05, 0.10, 0.20, 0.50]

F = build_sampled_basis_matrix_from_indices(
    time_mode="discrete",
    target_indices=TARGET_INDICES,
    thetas=THETAS,
)
Qf, _ = np.linalg.qr(F)
P_perp = np.eye(Qf.shape[0]) - Qf @ Qf.T
rng = np.random.default_rng(42)


In [4]:
def random_full_row_rank_matrix(rows: int, cols: int, seed: int) -> np.ndarray:
    rng_local = np.random.default_rng(seed)
    return rng_local.standard_normal((rows, cols))


def make_cases(alpha: float) -> dict[str, np.ndarray]:
    A = random_full_row_rank_matrix(Qf.shape[1], Qf.shape[1], seed=123)
    Z = rng.standard_normal((Qf.shape[0], Qf.shape[1]))
    Z_perp = P_perp @ Z
    return {
        "same_subspace": Qf @ A,
        "same_plus_orth_noise": np.concatenate([Qf @ A, alpha * Z_perp], axis=1),
        "random": rng.standard_normal((Qf.shape[0], Qf.shape[1])),
        "random_perp": P_perp @ rng.standard_normal((Qf.shape[0], Qf.shape[1])),
    }


In [5]:
records = []
case_specs = [(alpha, case_name, H) for alpha in ORTH_NOISE_SCALES for case_name, H in make_cases(alpha).items()]
for alpha, case_name, H in tqdm(case_specs, desc="experiment 1 cases"):
    metrics = calculate_subspace_alignment_from_matrices(H=H, F=F, theory_dim=2 * NUM_FREQS)
    singular_values = np.linalg.svd(H, full_matrices=False, compute_uv=False)
    rank_metrics = calculate_rank_metrics(singular_values, threshold=0.05)
    records.append({
        "alpha": alpha,
        "case": case_name,
        "rank_threshold": rank_metrics["rank_threshold"],
        "rank_entropy": rank_metrics["rank_entropy"],
        "h_numerical_dim": metrics["h_numerical_dim"],
        "align_coverage_full": metrics["align_coverage_full"],
        "align_purity_full": metrics["align_purity_full"],
        "align_coverage_top": metrics["align_coverage_top"],
        "align_mean_angle_deg_top": metrics["align_mean_angle_deg_top"],
        "recon_r2_qf_from_h": metrics["recon_r2_qf_from_h"],
    })

results_df = pd.DataFrame(records)
results_df.sort_values(["alpha", "case"]).reset_index(drop=True)


,alpha,case,align_coverage_full,align_purity_full,align_coverage_top,align_mean_angle_deg_top,recon_r2_qf_from_h
0,0.05,random,3.166402e-03,3.166402e-03,3.166402e-03,8.740921e+01,0.003166
1,0.05,random_perp,1.387973e-32,1.387973e-32,1.387973e-32,9.000000e+01,0.000000
2,0.05,same_plus_orth_noise,1.000000e+00,5.000000e-01,5.000000e-01,4.500000e+01,1.000000
3,0.05,same_subspace,1.000000e+00,1.000000e+00,1.000000e+00,6.181338e-07,1.000000
4,0.10,random,4.903682e-03,4.903682e-03,4.903682e-03,8.662164e+01,0.004904
5,0.10,random_perp,7.609154e-33,7.609154e-33,7.609154e-33,9.000000e+01,0.000000
6,0.10,same_plus_orth_noise,1.000000e+00,5.000000e-01,1.250000e-01,7.875000e+01,1.000000
7,0.10,same_subspace,1.000000e+00,1.000000e+00,1.000000e+00,6.181338e-07,1.000000
8,0.20,random,2.637950e-03,2.637950e-03,2.637950e-03,8.747067e+01,0.002638
9,0.20,random_perp,7.212444e-33,7.212444e-33,7.212444e-33,9.000000e+01,0.000000


In [ ]:
detailed_records = []
case_specs = [(alpha, case_name, H) for alpha in ORTH_NOISE_SCALES for case_name, H in make_cases(alpha).items()]
for alpha, case_name, H in tqdm(case_specs, desc="experiment 1 detailed metrics"):
    metrics = calculate_subspace_alignment_from_matrices(H=H, F=F, theory_dim=2 * NUM_FREQS)
    singular_values = np.linalg.svd(H, full_matrices=False, compute_uv=False)
    rank_metrics = calculate_rank_metrics(singular_values, threshold=0.05)
    row = {
        "alpha": alpha,
        "case": case_name,
        "rank_threshold": rank_metrics["rank_threshold"],
        "rank_entropy": rank_metrics["rank_entropy"],
    }
    for key, value in metrics.items():
        if np.isscalar(value):
            row[key] = value
    detailed_records.append(row)

detailed_results_df = pd.DataFrame(detailed_records)
detailed_results_df.sort_values(["alpha", "case"]).reset_index(drop=True)
